# DQMBot Batch Query
Loads API key from `.env`, sends a list of prompts to the FNAL OpenWebUI instance, and collects results into a DataFrame.

In [ ]:
# --- Install dotenv if needed (already present on EAF, but just in case) ---
# !pip install python-dotenv --quiet

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

# Load .env from same directory as this notebook
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))
API_KEY = os.environ.get('OWUI_API_KEY', '')

if not API_KEY:
    raise EnvironmentError('OWUI_API_KEY not found in .env')
print(f'API key loaded: {API_KEY[:8]}...')

API key loaded: sk-eb9aa...


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
OWUI_URL  = 'https://openwebui.fnal.gov/'   # ← replace with actual internal URL
MODEL     = ''                         # ← set model ID, or leave '' for server default
TIMEOUT   = 120                        # seconds per request
DELAY     = 1.0                        # seconds between requests (be polite)
# ──────────────────────────────────────────────────────────────────────────────

In [3]:
def list_models() -> list:
    """Return available model IDs on this OpenWebUI instance."""
    r = requests.get(
        f'{OWUI_URL}/api/models',
        headers={'Authorization': f'Bearer {API_KEY}'},
        timeout=30,
    )
    r.raise_for_status()
    return [m['id'] for m in r.json()['data']]

print('Available models:')
for m in list_models():
    print(f'  {m}')

Available models:
  835
  acnet-documentation
  acnet-documentation-coder
  dqm-chatbot
  gemma3:latest
  qwen2.5vl:32b
  qwen2.5vl:latest
  qwen3-vl:latest
  erlang-otp
  litellm-ow.qwen/qwen3-coder-next
  qwen3-vl:32b
  srf-data-test
  stockroom
  aeolus
  gpt-oss:20b
  qwen2.5:7b
  stockroom-clone-hayden
  stockroom-clone-hayden-medium
  litellm-ow.qwen/qwen35-9b
  qwenqwen35-9b-no-thinking
  nonfree.azure/gpt-5-nano
  nonfree.azure/auto
  vllm.gpt-oss:120b
  litellm-ow.google/gemma4-31b
  litellm-ow.qwen/qwen3.6
  policies
  nonfree.azure/gpt-5.4


In [8]:
def get_knowledge_map():
    r = requests.get(
        f"{OWUI_URL}/api/v1/knowledge/",
        headers={"Authorization": f"Bearer {API_KEY}"},
        timeout=30,
    )
    r.raise_for_status()
    return {kb["name"]: kb["id"] for kb in r.json()["items"]}

kb_map = get_knowledge_map()
# kb_map["ECFR"]  →  "01120bf5-3580-49d4-8bc5-4964b7d96cef"

In [9]:
kb_map

{'ECFR': '01120bf5-3580-49d4-8bc5-4964b7d96cef',
 'FESHM': '281b78b3-3cb2-4cfe-ab4b-9e7547713f82',
 'Fermilab Policies': 'ff41ce28-5f46-4f3a-a423-310fd63a21fd',
 'Rookie Books': 'c0de275c-458a-44f7-b800-834c6e52eb1a',
 'Erlang OTP-26.2.5.11': 'b18bde0e-da9c-45e6-8cbb-5f4fe7400b5d',
 'AEOLUS Codebase': 'cb71bb35-e24b-46f9-b31f-f946decb1fb5',
 'Stockroom': 'd55cb60f-05f9-4de0-a73e-a8f9417da34f',
 'Stockroom Catalog': '6bab354e-7671-43c4-868c-1baaf17c7491',
 'prime contract': 'f6573b12-8dce-4938-98a2-ccf8012f9487',
 'DQM shift rules': '2e286b31-0b40-44ff-8194-3733e5667c5a',
 'ACNET Documentation': 'c9d820b1-b7b6-45b8-83ea-d705a45cc32a',
 'procedures': 'c39ca83a-7b53-4b1c-8a04-3118b7fce09a',
 'Dosimetry': '322b627b-dd09-44c2-82df-81baded252ef',
 'Crosswalk': 'f88fa79e-3ada-498e-bdd2-53a8219770f0',
 'RP Procedures': '6c60d847-532b-4a80-914a-cc068864d7d1'}

In [ ]:
def query(prompt: str, system: str = '', model: str = MODEL) -> dict:
    """
    Send a single prompt. Returns a dict with keys:
      prompt, response, model_used, latency_s, error
    """
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})

    payload = {'messages': messages}
    if model:
        payload['model'] = model

    t0 = time.time()
    try:
        r = requests.post(
            f'{OWUI_URL}/api/chat/completions',
            headers={
                'Authorization': f'Bearer {API_KEY}',
                'Content-Type': 'application/json',
            },
            json=payload,
            timeout=TIMEOUT,
        )
        r.raise_for_status()
        data = r.json()
        return {
            'prompt':     prompt,
            'response':   data['choices'][0]['message']['content'],
            'model_used': data.get('model', model),
            'latency_s':  round(time.time() - t0, 2),
            'error':      None,
        }
    except Exception as e:
        return {
            'prompt':     prompt,
            'response':   None,
            'model_used': model,
            'latency_s':  round(time.time() - t0, 2),
            'error':      str(e),
        }

In [ ]:
# ── Prompts ───────────────────────────────────────────────────────────────────
# Edit this list with your actual DQM questions
SYSTEM_PROMPT = (
    'You are a CMS DQM expert assistant. '
    'Answer concisely and technically. '
    'If you are unsure, say so.'
)

PROMPTS = [
    'What does a hot strip in the SiStrip occupancy map indicate?',
    'Describe the typical signature of a ECAL supercrystal with a stuck ADC.',
    'What DQM alarm should fire when the CSC local trigger efficiency drops below 95%?',
    'How do you distinguish a noisy channel from a dead channel in the pixel detector DQM?',
    'What is the expected number of primary vertices per event at Run 3 pileup conditions?',
]
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Run batch ─────────────────────────────────────────────────────────────────
results = []
for i, prompt in enumerate(PROMPTS):
    print(f'[{i+1}/{len(PROMPTS)}] Querying...', end=' ', flush=True)
    result = query(prompt, system=SYSTEM_PROMPT)
    results.append(result)
    status = 'ERROR' if result['error'] else f'{result["latency_s"]}s'
    print(status)
    time.sleep(DELAY)

print('\nDone.')

In [ ]:
# ── Review results ────────────────────────────────────────────────────────────
df = pd.DataFrame(results)

# Wide display so responses aren't truncated
pd.set_option('display.max_colwidth', 300)
display(df[['prompt', 'response', 'latency_s', 'error']])

In [ ]:
# ── Print responses readably ──────────────────────────────────────────────────
for row in results:
    print('='*72)
    print(f'PROMPT:  {row["prompt"]}')
    print(f'LATENCY: {row["latency_s"]}s  |  MODEL: {row["model_used"]}')
    print()
    if row['error']:
        print(f'ERROR: {row["error"]}')
    else:
        print(row['response'])
    print()

In [ ]:
# ── Save to CSV (optional) ────────────────────────────────────────────────────
out_path = 'dqmbot_results.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')